In [ ]:
from typing import Iter
import numpy as np
import pyquist as pq


def iter_frames(audio: pq.Audio, hop_length: int, frame_length: int) -> Iter[np.ndarray]:
    for n in range(0, len(audio), hop_length):
        yield audio.samples[n:n + frame_length]


def overlap_add(frames: np.ndarray, hop_length: int, sample_rate: int):
    num_frames, frame_length, num_channels = frames.shape
    num_samples = hop_length * (num_frames - 1) + frame_length
    out = np.zeros((num_samples, num_channels), dtype=frames.dtype)
    for k, frame in enumerate(frames):
        out[k * hop_length:k * hop_length + frame_length] += frame
    return pq.Audio(out, sample_rate)

In [ ]:
# Extract frames and glue them back together. With a rectangular window and
# N_H = N_F (0% overlap) this is perfect reconstruction. Try N_H = N_F // 2
# (overlap, doubles the amplitude) or N_H = 2 * N_F (gaps) and listen!
audio = pq.Audio.from_file("../assets/audio-trio.wav")
N_F = 1024        # frame length (samples)
N_H = 1024        # hop length  (samples)

frames = np.array(iter_frames(audio, N_H, N_F))
print(frames.shape)
reconstructed = overlap_add(frames, N_H, audio.sample_rate)
pq.play(reconstructed)